# 05 — Tokenization
**Goal:** Master word, sentence, and subword tokenization.

Tokenization splits text into atomic units — **tokens** — and it is the first decision that shapes everything downstream: POS tagging, lemmatization, NER, and keyword matching all operate on whatever the tokenizer produced. "Word" is not a well-defined concept: contractions, abbreviations, emails, and `C++` each demand different treatment, and different tools draw the boundaries differently.

**Why it matters for resumes / ATS:** ATS keyword matching compares tokens against job-description terms. If the tokenizer splits `scikit-learn` into `scikit`, `-`, `learn` or mangles `C++`, the literal match against a JD keyword fails even though the skill is present. Tokenization is where resume-specific vocabulary (hyphens, slashes, plus signs, version numbers) meets the real world — and where naive whitespace splitting quietly loses matches.

## 1. Word Tokenization — Comparing Approaches

Three tokenizers, three different notions of "word". Python's `str.split()` cuts on whitespace only; NLTK's `word_tokenize` uses the Punkt model trained on general English; spaCy's tokenizer is a statistical model trained on web text — so it has seen emails, URLs, and contractions in the wild.

**What the code does:** after downloading the Punkt data (`nltk.download("punkt_tab", quiet=True)`), it tokenizes the same sentence three ways:
- whitespace split keeps `Don't` and `john.smith@email.com` whole, but leaves punctuation glued to words (`forget:`, `(work)`)
- NLTK splits the contraction (`Don't` → `Do`, `n't`) and the possessive (`Smith's` → `Smith`, `'s`), but breaks the email into `john.smith`, `@`, `email.com`
- spaCy handles the contraction identically, yet keeps the email address as one token

**Try it:** compare the two lists in the stored output — the email line is the differentiator: `john.smith@email.com` survives as a single token only in spaCy, which matters when resumes list `name@domain.com` contact lines.

In [1]:
import re, spacy, nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
nlp = spacy.load("en_core_web_sm")

text = "Don't forget: Dr. Smith's email is john.smith@email.com (work)"
print(f"Whitespace: {text.split()}")
print(f"NLTK:      {word_tokenize(text)}")
print(f"spaCy:     {[t.text for t in nlp(text)]}")

Whitespace: ["Don't", 'forget:', 'Dr.', "Smith's", 'email', 'is', 'john.smith@email.com', '(work)']
NLTK:      ['Do', "n't", 'forget', ':', 'Dr.', 'Smith', "'s", 'email', 'is', 'john.smith', '@', 'email.com', '(', 'work', ')']
spaCy:     ['Do', "n't", 'forget', ':', 'Dr.', 'Smith', "'s", 'email', 'is', 'john.smith@email.com', '(', 'work', ')']


## 2. spaCy Handles Resume Edge Cases

Resumes are dense with non-standard tokens: hyphenated library names, `C++`, slash-separated cloud stacks, "5+ years" ranges, city suffixes. How a tokenizer splits these determines whether downstream keyword matching ever sees the full string.

**What the code does:** parses five resume-typical tokens and prints the spaCy tokenization of each:
- `scikit-learn` → `['scikit', '-', 'learn']` — hyphens split, so the full library name is never a single token
- `C++` → `['C++']` — kept whole; the plus signs are not split off
- `AWS/GCP/Azure` → `['AWS', '/', 'GCP', '/', 'Azure']` — slashes separate, but each cloud name stays intact
- `5+ years` → `['5', '+', 'years']` and `New York-based` → `['New', 'York', '-', 'based']`

**Try it:** if your matcher uses exact token equality, `scikit-learn` will never match a JD line — plan for phrase-level matching (Ch. 13) or a custom matcher for hyphenated skills. `C++` staying whole is the pleasant surprise; do not assume it.

In [2]:
nlp = spacy.load("en_core_web_sm")
for t in ["scikit-learn", "C++", "AWS/GCP/Azure", "5+ years", "New York-based"]:
    print(f"'{t:20s}' -> {[tok.text for tok in nlp(t)]}")

'scikit-learn        ' -> ['scikit', '-', 'learn']
'C++                 ' -> ['C++']
'AWS/GCP/Azure       ' -> ['AWS', '/', 'GCP', '/', 'Azure']
'5+ years            ' -> ['5', '+', 'years']
'New York-based      ' -> ['New', 'York', '-', 'based']


## 3. Sentence Tokenization

Sentence boundaries look trivial and are not: periods inside abbreviations (`Mr.`, `Dr.`), interjections, and mixed punctuation all fool naive splitting. Sentence segmentation matters because downstream steps (bullet-level analysis, the SVO extraction of Ch. 11) assume each sentence is one unit.

**What the code does:** runs NLTK's `sent_tokenize` and spaCy's `doc.sents` on the same string containing `Mr.`, `Dr.`, `!`, and `?`:
- NLTK returns 3 sentences, correctly refusing to split after `Mr.` and `Dr.`
- spaCy returns the identical 3 sentences — both tools learned that an abbreviation period is not a sentence end

**Try it:** the stored outputs line up exactly: sentence 1 ends at `2020.`, sentence 2 at `promotion!`, sentence 3 at `enough?`. On resume text, sentence counts usually equal bullet counts — a quick sanity metric for later parsing stages.

In [3]:
from nltk.tokenize import sent_tokenize
text = "Mr. Smith joined Google in 2020. Dr. Jones approved his promotion! Was it enough?"
for i, s in enumerate(sent_tokenize(text)):
    print(f"{i+1}. {s}")
print("\nspaCy:")
for i, s in enumerate(nlp(text).sents):
    print(f"{i+1}. {s.text}")

1. Mr. Smith joined Google in 2020.
2. Dr. Jones approved his promotion!
3. Was it enough?

spaCy:
1. Mr. Smith joined Google in 2020.
2. Dr. Jones approved his promotion!
3. Was it enough?


## 4. Subword Tokenization (BPE / WordPiece)

Modern transformer models do not tokenize into words at all. **WordPiece** (used by BERT) splits rare or unseen words into frequent subword fragments bounded by a fixed vocabulary; continuations are marked with `##`. The trade-off: near-infinite coverage of novel words at the cost of tokens that no longer correspond to dictionary words.

**What the code does:** loads the `bert-base-uncased` tokenizer via `transformers.AutoTokenizer.from_pretrained(...)` and tokenizes four resume-relevant words:
- `embeddings` → `['em', '##bed', '##ding', '##s']`
- `tokenization` → `['token', '##ization']`
- `scikit-learn` → `['sci', '##kit', '-', 'learn']`
- `TensorFlow` → `['tensor', '##flow']` — uncased, so the capital T is dropped

**Try it:** the first-run noise (PyTorch-version warning, HF Hub download progress bars) is harmless — the tokenizer downloads its vocab files on first use. The takeaway: subword tokens are great for LLM embeddings, but keyword matching against JDs still wants the whole-word view from §1–2.

In [4]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
for w in ["embeddings", "tokenization", "scikit-learn", "TensorFlow"]:
    print(f"'{w}' -> {tok.tokenize(w)}")

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

'embeddings' -> ['em', '##bed', '##ding', '##s']
'tokenization' -> ['token', '##ization']
'scikit-learn' -> ['sci', '##kit', '-', 'learn']
'TensorFlow' -> ['tensor', '##flow']


## Summary: Choose spaCy for resume tokenization. It handles domain-specific tokens (C++, hyphens, slashes) correctly.

Whitespace splitting glues punctuation to words, NLTK splits emails and some contractions, and BPE subwords are built for embeddings, not matching. spaCy's statistical tokenizer keeps emails whole, handles `C++` and slashes sensibly, and produces tokens that align with how skills are actually written — the right default for resume processing. Hyphenated skills (`scikit-learn`) remain the known gap and need phrase-level handling later.

Tokenization is the foundation everything else stands on: in production, Ch. 06 normalization cleans text *before* it reaches the tokenizer, and the tokens produced here are exactly what POS tagging (Ch. 10) and NER (Ch. 12) will label.